### SoRL Playground

This notebook demonstrates the SoRL post-training pipeline. Porting a OSS model, and adopt SoRL trainer to post-train the model accordingly. 

In [1]:
# Setup
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

# Disable MPS for stability
if hasattr(torch.backends, 'mps'):
    torch.backends.mps.is_available = lambda: False

from transformers import TrainingArguments, AutoTokenizer
from sorl.sorl_wrapper import SorlModelWrapper
from sorl.sorl_trainer import SorlTrainer

device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


In [2]:
# Initialize SoRL model
model_name = "Qwen/Qwen2.5-0.5B"
model = SorlModelWrapper.from_pretrained(
    model_name,
    abstract_vocab_size_list=[128],
    memory_span=1792
)
model = model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name) 

Some weights of Qwen2ForCausalLM were not initialized from the model checkpoint at Qwen/Qwen2.5-0.5B and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [3]:
# We might be missing the "mask" gadget in the trainer, if we want to 
# use GSM8K alike dataset --- is there a build-in approach for it? 


# Replace your SimpleDataset with GSM8K data loading
from datasets import load_dataset
 
# Load GSM8K dataset
gsm8k_dataset = load_dataset("gsm8k", "main", split="train")
 
# Create dataset class for GSM8K
class GSM8KDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, tokenizer, max_length=16):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        example = self.dataset[idx]
        
        # Format for math reasoning
        text = f"Question: {example['question']}\nAnswer: {example['answer']}"
        
        # Tokenize
        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        
        return {
            "input_ids": encoded["input_ids"].squeeze(),
            "attention_mask": encoded["attention_mask"].squeeze()
        }
 
# Use GSM8K dataset
train_dataset = GSM8KDataset(gsm8k_dataset, tokenizer)

# Training arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./sorl_results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    warmup_steps=1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_steps=10,
    eval_steps=10,
)

# Create SoRL trainer
print("Creating SoRL trainer...")
trainer = SorlTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    num_rollouts=4,
    K=3,
    max_iterations=2,
    memory_span_abs=1792,
    memory_span_traj=1792,
    temperature=1.0,
    alpha_info_gain=10.0,
    alpha_abs=0.1,
    alpha_soft_zipf=1.0,
)

print("Trainer created successfully!")
print(f"Model vocab sizes: {model.vocab_sizes}")
print(f"Total vocab size: {model.total_vocab_size}")

Creating SoRL trainer...
Trainer created successfully!
Model vocab sizes: tensor([151936,    129])
Total vocab size: 152065


/Users/ksgk/Implementation/mod_gpt/sorl/sorl_trainer.py:222: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SorlTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


In [4]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
wandb: Currently logged in as: fangyuan-yu18 (ksgk-hack) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/Users/ksgk/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[DEBUG] Custom training_step called


Step,Training Loss
10,7.066700


[DEBUG] Custom training_step called
[DEBUG] Custom training_step called
[DEBUG] Custom training_step called
[DEBUG] Custom training_step called
[DEBUG] Custom training_step called
[DEBUG] Custom training_step called
[DEBUG] Custom training_step called
[DEBUG] Custom training_step called
[DEBUG] Custom training_step called
[DEBUG] Custom training_step called
[DEBUG] Custom training_step called


KeyboardInterrupt: 

In [ ]:
# --- manual training loop works ---
model.train()

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
dataloader = trainer.get_train_dataloader()

for i, batch in enumerate(dataloader):
    batch = {k: v.to(device) for k, v in batch.items()}
    
    optimizer.zero_grad()
    loss = trainer.compute_loss(model, batch)
    loss.backward()
    optimizer.step()
    
    print(f"Step {i}: loss={loss.item():.4f}")
    
    if i >= 5:
        break

print("Manual training loop completed!")

In [4]:
# --- but folded training collapse with no error message ---
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
wandb: Currently logged in as: fangyuan-yu18 (ksgk-hack) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/Users/ksgk/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,-2.774200
20,14.920300
30,11.303000
40,11.428100
50,10.003300
60,9.443300
70,8.898900
80,8.362800
90,8.237800
100,8.587200


RuntimeError: [enforce fail at inline_container.cc:664] . unexpected pos 4497748608 vs 4497748496

In [ ]:
from sorl.sorl_trainer import sorl_search, SoRLLoss
from datasets import load_dataset
from torch.utils.data import DataLoader

# Load GSM8K dataset
dataloader = DataLoader(train_dataset, batch_size=3, collate_fn=lambda x: {
    'input_ids': torch.stack([item['input_ids'] for item in x]),
    'attention_mask': torch.stack([item['attention_mask'] for item in x])
})
batch = next(iter(dataloader))

input_ids = batch['input_ids'].to(device)
attention_mask = batch["attention_mask"].to(device)
pad_token_id = tokenizer.pad_token_id
     
n = 2
K = 4
max_iterations = 2 
memory_span_abs = 512
memory_span_traj = 512
temperature = 1.0
tokens = input_ids
pad_token_id = tokenizer.pad_token_id

# ---- sorl search ----
best_data, best_ppt, best_ppt_advantage, expanded_attention_mask = sorl_search(model, 
            input_ids, 
            attention_mask, 
            pad_token_id, 
            n, K, max_iterations, memory_span_abs, memory_span_traj, temperature)

# ---- loss computation ---
loss_fn = SoRLLoss(abs_vocab_size=model.vocab_sizes[-1])

input_ids = best_data
attention_mask = expanded_attention_mask

labels = input_ids.clone()
labels[attention_mask == 0] = -100

outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels, memory_span_abs=memory_span_abs, memory_span_traj=memory_span_traj)
base_traj_loss = outputs.loss

# ----- SoRL loss propagation -----
loss = loss_fn(input_ids, model, base_traj_loss.detach(), attention_mask, memory_span_abs, memory_span_traj)
total_loss = base_traj_loss + loss[0] + loss[1] + loss[2]
total_loss.backward()  # Does this crash?